# OpenAI Function Calling In LangChain

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o")

In [2]:
from typing import List
from pydantic import BaseModel, Field

## Pydantic Syntax

Pydantic data classes are a blend of Python's data classes with the validation power of Pydantic. 

They offer a concise way to define data structures while ensuring that the data adheres to specified types and constraints.

In standard python you would create a class like this:

In [3]:
class User:
    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email

In [4]:
foo = User(name="Joe",age=32, email="joe@gmail.com")

In [5]:
foo.name

'Joe'

In [6]:
foo = User(name="Joe",age="bar", email="joe@gmail.com")

In [7]:
foo.age

'bar'

In [8]:
class pUser(BaseModel):
    name: str
    age: int
    email: str

In [9]:
foo_p = pUser(name="Jane", age=32, email="jane@gmail.com")

In [10]:
foo_p.name

'Jane'

<p style=\"background-color:#F5C780; padding:15px\"><b>Note:</b> The next line is expected to fail.</p>

In [11]:
foo_p = pUser(name="Jane", age="bar", email="jane@gmail.com")

ValidationError: 1 validation error for pUser
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='bar', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

In [12]:
class Class(BaseModel):
    students: List[pUser]

In [13]:
obj = Class(
    students=[pUser(name="Jane", age=32, email="jane@gmail.com")]
)

In [14]:
obj

Class(students=[pUser(name='Jane', age=32, email='jane@gmail.com')])

## Pydantic to OpenAI function definition


In [15]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str = Field(description="airport code to get weather for")

In [16]:
from langchain_core.utils.function_calling import convert_to_openai_tool

In [17]:
weather_tool = convert_to_openai_tool(WeatherSearch)

In [18]:
weather_tool

{'type': 'function',
 'function': {'name': 'WeatherSearch',
  'description': 'Call this with an airport code to get the weather at that airport',
  'parameters': {'properties': {'airport_code': {'description': 'airport code to get weather for',
     'type': 'string'}},
   'required': ['airport_code'],
   'type': 'object'}}}

In [19]:
class WeatherSearch1(BaseModel):
    airport_code: str = Field(description="airport code to get weather for")

<p style=\"background-color:#F5C780; padding:15px\"><b>Note:</b> The next cell is expected to generate an error.</p>

In [20]:
convert_to_openai_tool(WeatherSearch1)

{'type': 'function',
 'function': {'name': 'WeatherSearch1',
  'description': '',
  'parameters': {'properties': {'airport_code': {'description': 'airport code to get weather for',
     'type': 'string'}},
   'required': ['airport_code'],
   'type': 'object'}}}

In [21]:
class WeatherSearch2(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str

In [22]:
convert_to_openai_tool(WeatherSearch2)

{'type': 'function',
 'function': {'name': 'WeatherSearch2',
  'description': 'Call this with an airport code to get the weather at that airport',
  'parameters': {'properties': {'airport_code': {'type': 'string'}},
   'required': ['airport_code'],
   'type': 'object'}}}

In [23]:
from langchain_openai import ChatOpenAI

In [24]:
model = ChatOpenAI()

In [25]:
model.bind_tools([WeatherSearch]).invoke("what is the weather in SF today?")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 70, 'total_tokens': 87, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSb9aCDVUtZlxFuzezsfPFpp3G9zQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7077-7dcd-7272-98ed-108f408b918e-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_UBgqvq5xWOfQRzFCqUf5LQKO', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 17, 'total_tokens': 87, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [26]:
model_with_tools = model.bind_tools([WeatherSearch])

In [27]:
model_with_tools.invoke("what is the weather in sf?")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 69, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSb9hUJduMZPrr8oz3e6Sz38gpugS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7077-9ace-7273-b06d-9b641e091978-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_vlzOO57BQDZqt0AJ89dgPurS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 17, 'total_tokens': 86, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Forcing it to use a function

We can force the model to use a function

In [28]:
model_with_forced_tool = model.bind_tools(
    [WeatherSearch],
    tool_choice={"type": "function", "function": {"name": "WeatherSearch"}}
)

In [29]:
model_with_forced_tool.invoke("what is the weather in sf?")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 79, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSb9uv6Gf9NIi5VehiVBvFGaqiMun', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7077-ceda-7c23-ab32-9654537bb1ee-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_oSQOqAGc3Nd2llchT6UVLeS6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 7, 'total_tokens': 86, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [30]:
model_with_forced_tool.invoke("hi!")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 74, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSb9y4IANgBRxJmg3coFo6zOL20tp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7077-dc0c-71d1-aa06-98e91b5f4324-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_41PXKJdSEBCULWwT4l6xmEag', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 7, 'total_tokens': 81, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Using in a chain

We can use this model bound to function in a chain as we normally would

In [31]:
from langchain_core.prompts import ChatPromptTemplate

In [32]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "{input}")
])

In [33]:
chain = prompt | model_with_tools

In [34]:
chain.invoke({"input": "what is the weather in sf?"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 75, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSbANF27WiVJXzvR9U2xIPR2uF1U8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7078-4093-74b2-989b-31f5dfbd2d1b-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_5OKnh0M75hiGZylUV6hSZqtB', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 17, 'total_tokens': 92, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Using multiple functions

Even better, we can pass a set of function and let the LLM decide which to use based on the question context.

In [35]:
class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""
    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results")

In [36]:
tools = [
    convert_to_openai_tool(WeatherSearch),
    convert_to_openai_tool(ArtistSearch),
]

In [37]:
model_with_tools = model.bind_tools([WeatherSearch, ArtistSearch])

In [38]:
model_with_tools.invoke("what is the weather in sf?")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 116, 'total_tokens': 133, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSbAr0eT0y9BYq3aB8L9lkbCJR1TU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7078-b39c-7722-b602-ee0a79dcc476-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_euQ0RyO3K1ERJZnQM2Ue14bD', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 116, 'output_tokens': 17, 'total_tokens': 133, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [39]:
model_with_tools.invoke("what are three songs by taylor swift?")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 118, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSbAuO0crzdI2VxC4krbm33yZBtC7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7078-bd16-7781-aa13-c1e3264d7027-0', tool_calls=[{'name': 'ArtistSearch', 'args': {'artist_name': 'Taylor Swift', 'n': 3}, 'id': 'call_u35ksCxbZ8Rznf9Dc9NMMSOZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 21, 'total_tokens': 139, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [40]:
model_with_tools.invoke("hi!")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 111, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSbAwKdQqW8eA2erHQzqOF6ALgwkZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7078-c8b0-7dd2-b9f6-7a2d2591b98b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 10, 'total_tokens': 121, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})